# Food.com Dataset

This notebook cleans and analyzes the Food.com dataset from Kaggle:
https://www.kaggle.com/datasets/irkaal/foodcom-recipes-and-reviews

## Load Data

In [1]:
import pandas as pd

data = pd.read_parquet("data/recipes.parquet")
print("total number of samples:", len(data))

total number of samples: 522517


## View Single Sample

In [2]:
data.iloc[0]

RecipeId                                                                   38.0
Name                                          Low-Fat Berry Blue Frozen Dessert
AuthorId                                                                   1533
AuthorName                                                               Dancer
CookTime                                                                  PT24H
PrepTime                                                                  PT45M
TotalTime                                                              PT24H45M
DatePublished                                         1999-08-09 21:46:00+00:00
Description                   Make and share this Low-Fat Berry Blue Frozen ...
Images                        [https://img.sndimg.com/food/image/upload/w_55...
RecipeCategory                                                  Frozen Desserts
Keywords                      [Dessert, Low Protein, Low Cholesterol, Health...
RecipeIngredientQuantities              

## Investigate Dataset Values

Print important stats about selected dataset columns

In [3]:
kwrds = data['Keywords'].explode().unique()
catg = data['RecipeCategory'].unique()
print("found", len(catg), "categories:\n", catg[:20])
print("--------------------------------------------")
print("found", len(kwrds), "keywords:\n", kwrds[:20])
print("--------------------------------------------")
print("maximum rating:", data["AggregatedRating"].max())
print("maximum review count:", data["ReviewCount"].max())
print("average review count:", data["ReviewCount"].mean())

found 312 categories:
 ['Frozen Desserts' 'Chicken Breast' 'Beverages' 'Soy/Tofu' 'Vegetable'
 'Pie' 'Chicken' 'Dessert' 'Southwestern U.S.' 'Sauces' 'Stew'
 'Black Beans' '< 60 Mins' 'Lactose Free' 'Weeknight' 'Yeast Breads'
 'Whole Chicken' 'High Protein' 'Cheesecake' 'Free Of...']
--------------------------------------------
found 315 keywords:
 ['Dessert' 'Low Protein' 'Low Cholesterol' 'Healthy' 'Free Of...' 'Summer'
 'Weeknight' 'Freezer' 'Easy' 'Chicken Thigh & Leg' 'Chicken' 'Poultry'
 'Meat' 'Asian' 'Indian' 'Stove Top' '< 60 Mins' 'Beans' 'Vegetable'
 'Broil/Grill']
--------------------------------------------
maximum rating: 5.0
maximum review count: 3063.0
average review count: 5.227784080166383


## Discuss Dataset Values
- rating system is 0-5.
- average number of reviews is 29.
- we see a large number of categories (224) and keywords (263)
- we see repeating values like: Chicken, Dessert, Vegetable, Protein

### Assumption
many of the 224 & 263 category and keyword values repeat and therefore redundant. <br> Let's explore it further.

In [4]:
# Find words that repeat as categories and keywords
intersection = set(catg).intersection(set(kwrds))

print("number of identical words:", len(intersection))
print("% of redundant categories:", len(intersection) / len(catg))
print("% of redundant keywords:", len(intersection) / len(kwrds))

number of identical words: 279
% of redundant categories: 0.8942307692307693
% of redundant keywords: 0.8857142857142857


### Discuss Assumption

Our assumption was correct - 77% of keywords and 91% of the categories are redundant.

## Filter Data 

### Filter Low Quality Receipes

Before we move on with consolidating the categories - let's get rid of bad or unvetted receipes. <br>
We will remove receipes with less than 15 reviews, as well as receipes with rating below 4.

In [5]:
data = data[(data['ReviewCount']>=15) & (data['AggregatedRating']>=4)].copy()
print("remaining number samples:", len(data))

remaining number samples: 15754


### Filter Missing Images

We will also remove images that result in loading errors.

#### Review Single Image Field

But first, we must see what a random image field from our dataset looks like, and what it stores

In [6]:
data.iloc[0]["Images"]

array(['https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/44/picsSKvFd.jpg'],
      dtype=object)

#### Dicuss Single Image Field 

We see that each image field stores a Numpy array rather than a link to a single image. We must take that into account in our filtering logic below.

### Define Check Images Function
In the following function, we will check only if the first image in the array exists.
<br>
As long as we have one good image - we keep the sample.

In [7]:
def check_image(image_array):
    """
    check if the first URL from a dataset image field loads or returns an error
    """
    if len(image_array) == 0:
        return False
    try:
        if requests.get(image_array[0], timeout=3).status_code == 200:
            return True
        else:
            return False
    except Exception:
        return False

### Call Check Images Function with Parallel Apply

In the following cell, we will force all available CPU cores to process the check images function. Instead of using the Pandas `apply()` method - we will use `parallel_apply`. This will reduce the processing time significantly.

In [8]:
from pandarallel import pandarallel
import requests

pandarallel.initialize()

%time data["1st_image_exists"] = data['Images'].parallel_apply(check_image)
print("Existing Images", sum(data["1st_image_exists"]), "/", len(data))

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
CPU times: user 62.3 ms, sys: 139 ms, total: 202 ms
Wall time: 7min 19s
Existing Images 14757 / 15754


### Remove Samples with Missing Images

We are left with 14757 images out of the initial 15754, filtering approx 1000 samples with missing images (or more accuratley - first image missing).
We will run the following line of code to facilitate it.

In [48]:
data = data[data["1st_image_exists"] == True].copy()
data = data.drop(columns=["1st_image_exists"])
len(data)

14757

## Keywords & Category Consolidation

### Pull Unique Keywords/Categories into a New Column
In the next cell we will consolidate keywords with categories to avoid redundancy and repetition. We will create a new column named "Tags" where a combination of both columns will be stored.

In [10]:
import numpy as np

# filter non-food-reelated categories (discovered with Gemini)
garbage_tags = {'household cleaner', 'bath/beauty', 'homeopathy/remedies', 'none'}

def get_unique_tags(row):
    k_list = row['Keywords'] if isinstance(row['Keywords'], (list, np.ndarray)) else []
    c_val = [row['RecipeCategory']] if pd.notna(row['RecipeCategory']) else []
    combined = {str(t).strip().lower() for t in (list(k_list) + c_val) if t}
    
    return list(combined - garbage_tags)

# consolidate in a new row
data['Tags'] = data.apply(get_unique_tags, axis=1)

print(data[['Tags']].head())

                                                 Tags
6                 [poultry, chicken, meat, < 60 mins]
11  [oven, european, very low carbs, chicken breas...
16   [oven, dessert, vegetable, < 4 hours, weeknight]
18              [oven, dessert, < 4 hours, pie, easy]
24  [summer, vegetable, < 30 mins, beans, free of....


### Remove Old Keyword & Categories Columns

In [16]:
data = data.drop(columns=['RecipeCategory', 'Keywords'])

## Fetch Meaningful Features
In the next cell, we will fetch 5 filters from the data into separate boolean columns.
<br>
We will check for diatary and lifestyle restrictions to make data search easier. (instead of searching trough a list of strings within the "Tags" column of each sample - we will search for a boolean value in a dedicated column).
<br>
The following columns will be added:
- is_vegetarian
- is_pork
- is_dairy_free
- is_low_calorie
- is_quick
  
### Search and Extract Meningful Features

In [13]:
import isodate

# flatten ingredients
ingred_str = data['RecipeIngredientParts'].str.join(" ").str.lower()

# pork ingredients (detected by Gemini)
pork_explicit = [
    'pork', 'bacon', 'ham', 'lard', 'prosciutto', 'chorizo', 'pancetta', 
    'mortadella', 'lardons', 'guanciale', 'cerdo', 'hocks', 'kielbasa'
]

# if contains pork...
data['is_pork'] = ingred_str.str.contains('|'.join(pork_explicit)).fillna(False)

# meat ingredients (non-vegetarian)
meat_fish_list = pork_explicit + [
    'bacon', 'chicken', 'beef', 'steak', 'turkey', 'lamb', 
    'fish', 'anchov', 'tuna', 'salmon', 'shrimp', 'crab', 'gelatin'
]

# if not meat...
data['is_vegetarian'] = ~ingred_str.str.contains('|'.join(meat_fish_list)).fillna(True)

# dairy ingredients (detected by Gemini)
dairy_list = ['milk', 'cheese', 'butter', 'cream', 'yogurt', 'whey', 'ghee', 'lard']

# if not dairy...
data['is_dairy_free'] = ~ingred_str.str.contains('|'.join(dairy_list)).fillna(True)

# parse ISO 8601 time strings to total minutes (Tactical Realism: don't trust the tags!)
def parse_time(time_str):
    try: return isodate.parse_duration(time_str).total_seconds() / 60
    except: return 0

total_time = data['CookTime'].apply(parse_time) + data['PrepTime'].apply(parse_time)

# if under 30 mins...
data['is_quick'] = total_time <= 30

# if under 400 calories...
data['is_low_calorie'] = data['Calories'] < 400

# cast on new columns
new_cols = ['is_pork', 'is_vegetarian', 'is_dairy_free', 'is_quick', 'is_low_calorie']
data[new_cols] = data[new_cols].astype(bool)

data.head()

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,...,RecipeServings,RecipeYield,RecipeInstructions,1st_image_exists,Tags,is_pork,is_vegetarian,is_dairy_free,is_quick,is_low_calorie
6,44.0,Warm Chicken A La King,1596,Joan Edington,PT3M,PT35M,PT38M,1999-09-17 04:47:00+00:00,I copied this one out of a friend's book so ma...,[https://img.sndimg.com/food/image/upload/w_55...,...,2.0,None,"[Melt 1 1/2 ozs butter, add the flour and cook...",True,"[poultry, chicken, meat, < 60 mins]",False,False,False,False,False
11,49.0,Chicken Breasts Lombardi,174711,Queen Dragon Mom,PT30M,PT45M,PT1H15M,1999-08-14 19:58:00+00:00,Make and share this Chicken Breasts Lombardi r...,[https://img.sndimg.com/food/image/upload/w_55...,...,6.0,None,[Cook mushrooms in 2 tbsp butter in a large s...,True,"[oven, european, very low carbs, chicken breas...",False,False,False,False,False
16,54.0,Carrot Cake,1535,Marg CaymanDesigns,PT50M,PT45M,PT1H35M,1999-09-13 15:20:00+00:00,This is one of the few recipes my husband ever...,[https://img.sndimg.com/food/image/upload/w_55...,...,12.0,1 bundt,"[Beat together the eggs, oil, and white sugar....",True,"[oven, dessert, vegetable, < 4 hours, weeknight]",False,True,False,False,False
18,56.0,Buttermilk Pie,1581,thefensk,PT1H,PT20M,PT1H20M,1999-08-30 10:02:00+00:00,This recipe was originally noted by my wife on...,[https://img.sndimg.com/food/image/upload/w_55...,...,8.0,None,"[Preheat oven to 400°F., Beat the butter and s...",True,"[oven, dessert, < 4 hours, pie, easy]",False,True,False,False,True
24,62.0,"Black Bean, Corn, and Tomato Salad",1570,Dave Miner,PT15M,PT10M,PT25M,1999-08-19 05:12:00+00:00,"This is easy, delicious, colorful, delicious, ...",[],...,2.0,None,"[In a bowl whisk together lemon juice, oil, an...",False,"[summer, vegetable, < 30 mins, beans, free of....",False,True,True,True,False


### Verfiy Results For Each Category

In [37]:
print("quick recipes:", len(data[data['is_quick'] == True]))
print("low_calorie recipes:", len(data[data['is_low_calorie'] == True]))
print("dairy-free recipes:", len(data[data['is_dairy_free'] == True]))
print("pork-free recipes:", len(data[data['is_pork'] == False]))
print("vegeterian recipes:", len(data[data['is_vegetarian'] == True]))

quick recipes: 6455
low_calorie recipes: 10076
dairy-free recipes: 6171
pork-free recipes: 14456
vegeterian recipes: 10535


## Save Processed Data Back to Parquet
We will save a copy of the processed data to a Parquet file for further exploration and analysis.

In [49]:
data.to_parquet("reviews_processed.parquet", index=False)

## Convert to JSON
We will also save the processed data into a JSON file for easy integration in APIs and software.

In [50]:
data.to_json("full_spoon.json", orient="records", indent=4)

## Check JSON
Let's double-check that the JSON file was saved properly

In [51]:
import json

# Read just the first record from the file you just created
with open("full_spoon.json", "r") as f:
    sample_data = json.load(f)
    print(json.dumps(sample_data[430], indent=4))

{
    "RecipeId": 8589.0,
    "Name": "Orange Julius",
    "AuthorId": 5811,
    "AuthorName": "Robbie Rice",
    "CookTime": null,
    "PrepTime": "PT15M",
    "TotalTime": "PT15M",
    "DatePublished": 982238580000,
    "Description": "Make and share this Orange Julius recipe from Food.com.",
    "Images": [
        "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/85/89/piciyaM7f.jpg",
        "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/85/89/picWj4rkY.jpg",
        "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/85/89/85b6C4EsTBusjBuPjKqC_image.jpg",
        "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/85/89/picmZqcKq.jpg",
        "https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/85/89/picSG0LSW.jpg"
    ],
    "RecipeIngredientQuantities": [
    

In [ ]:
!pip install pandarallel
!pip install ipywidgets
!pip install isodate